In [1]:
from mpramnist.Vaishnav2024 import VaishnavDataset
from mpramnist.Vaishnav2024 import LitModel_Vaishnav

import mpramnist.transforms as t

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1024
NUM_WORKERS = 16

length = 110
plasmid = VaishnavDataset.PLASMID.upper()
insert_start = plasmid.find("N" * 80)
right_flank = VaishnavDataset.RIGHT_FLANK
left_flank = plasmid[insert_start - length : insert_start]

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Important note**: Sequence lengths vary. To standardize them:

* Original flanks should be preserved

* Missing regions need to be supplemented from the source plasmid, use LeftFlank for it

* All sequences must be adjusted to the default 110 bp length (as in the original protocol)

In [2]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(left_flank, right_flank),
        t.LeftCrop(length, length),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
val_test_transform = t.Compose(
    [
        t.AddFlanks(left_flank, right_flank),
        t.LeftCrop(length, length),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

In the original study, two complementary environments with opposing selective pressures on URA3 gene expression (encoding an enzyme responsible for uracil synthesis) were investigated:

`defined` environment, where organismal fitness increases with gene expression (up to saturation);

`complex` environment + 5-FOA, where fitness decreases with Ura3p expression.

Use the `dataset_env_type` parameter to select either `'defined'` or `'complex'`.

# Dataset Specifications:

`defined`: (1) Contains 20 million sequences (2) 10% allocated for validation (3) Remainder used for training

`complex`: (1) Contains 31 million sequences (2) 10% allocated for validation (3) Remainder used for training

# Train **defined** env type

In [3]:
# load the data
train_dataset = VaishnavDataset(split="train",dataset_env_type="defined",transform=train_transform,root="../data/",)
val_dataset = VaishnavDataset(split="val",dataset_env_type="defined",transform=val_test_transform,root="../data/",)

print(len(train_dataset), len(val_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

18933667 2103740


In [4]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[1, 1, 1, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_defined = LitModel_Vaishnav(model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

# Train the model
trainer.fit(seq_model_defined, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.290  

Epoch 0:   0%|          | 2/18490 [00:01<3:35:56,  1.43it/s, v_num=70]

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 0: 100%|██████████| 18490/18490 [39:31<00:00,  7.80it/s, v_num=70]
-------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 3.76220 | Val Pearson: 0.88423 | Train Pearson: 0.84357 
-------------------------------------------------------------------------------

Epoch 0: 100%|██████████| 18490/18490 [40:18<00:00,  7.65it/s, v_num=70, val_loss=3.760, val_pearson=0.884, train_loss=5.000]

Metric val_loss improved. New best score: 3.762
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 18490/18490 [40:18<00:00,  7.64it/s, v_num=70, val_loss=3.760, val_pearson=0.884, train_loss=5.000]


In [5]:
def meaned_prediction(forw, rev, trainer, seq_model, name, is_paired=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print(name + " Pearson correlation")

    if is_paired:
        y_preds_forw_alt = torch.cat(
            [pred["alt_predicted"] for pred in predictions_forw]
        )
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets)

    return pears(mean_forw, targets)

## Test Sequences:

Test sequences are divided into three categories per environment:

* Reference (`native`)

* Alternative (`drift`)

* Paired (`paired`)

Use the `test_dataset_type`

In [6]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="native",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="native",transform=rev_transform,root="../data/",)

print("native info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="native")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


native info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 3978
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 18933667, 'val': 2103740, 'test native': 3978, 'test drift': 2986, 'test paired': 2986}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 39.10it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 56.27it/s]
native Pearson correlation


tensor(0.9759)

In [7]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="drift",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="drift",transform=rev_transform,root="../data/",)

print("drift info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="drift")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


drift info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 2986
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 18933667, 'val': 2103740, 'test native': 3978, 'test drift': 2986, 'test paired': 2986}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 36.71it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 32.01it/s]
drift Pearson correlation


tensor(0.9859)

### Note on paired sequences:

Each paired sequence contains: (1) A reference sequence (2) An alternative sequence (3) Differential expression column

In [8]:
dataset_paired = VaishnavDataset(
    split="test",
    dataset_env_type="defined",
    test_dataset_type="paired",
    root="../data/",
)
dataset_paired[0]

({'seq': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT',
  'seq_alt': 'CTTTAAATTCGGTGGGGACGCGTCGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT'},
 tensor(0.6423))

In [9]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="paired",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="paired",transform=rev_transform,root="../data/",)

print("paired info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="paired")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


paired info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 2986
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 18933667, 'val': 2103740, 'test native': 3978, 'test drift': 2986, 'test paired': 2986}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 21.02it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 22.93it/s]
paired Pearson correlation


tensor(-0.1663)

# Train **complex** env type

In [10]:
# load the data
train_dataset = VaishnavDataset(split="train",dataset_env_type="complex",transform=train_transform,root="../data/",)
val_dataset = VaishnavDataset(split="val",dataset_env_type="complex",transform=val_test_transform,root="../data/",)

print(len(train_dataset), len(val_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

28214427 3134936


In [11]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[1, 1, 1, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_complex = LitModel_Vaishnav(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=1
)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

# Train the model
trainer.fit(seq_model_complex, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.290 

Epoch 0: 100%|██████████| 27554/27554 [34:44<00:00, 13.22it/s, v_num=71]
-------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 5.25012 | Val Pearson: 0.84195 | Train Pearson: 0.80514 
-------------------------------------------------------------------------------

Epoch 0: 100%|██████████| 27554/27554 [35:35<00:00, 12.90it/s, v_num=71, val_loss=5.250, val_pearson=0.842, train_loss=6.360]

Metric val_loss improved. New best score: 5.250
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 27554/27554 [35:35<00:00, 12.90it/s, v_num=71, val_loss=5.250, val_pearson=0.842, train_loss=6.360]


## Test Sequences:

Test sequences are divided into three categories per environment:

* Reference (`native`)

* Alternative (`drift`)

* Paired (`paired`)

Use the `test_dataset_type`

In [12]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="native",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="native",transform=rev_transform,root="../data/",)

print("native info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="native")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


native info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 3929
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 28214427, 'val': 3134936, 'test native': 3929, 'test drift': 2983, 'test paired': 2983}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 38.90it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 54.35it/s]
native Pearson correlation


tensor(0.9646)

In [13]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="drift",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="drift",transform=rev_transform,root="../data/",)

print("drift info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="drift")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


drift info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 2983
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 28214427, 'val': 3134936, 'test native': 3929, 'test drift': 2983, 'test paired': 2983}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 31.85it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 51.98it/s]
drift Pearson correlation


tensor(0.9711)

### Note on paired sequences:

Each paired sequence contains: (1) A reference sequence (2) An alternative sequence (3) Differential expression column

In [14]:
dataset_paired = VaishnavDataset(
    split="test",
    dataset_env_type="complex",
    test_dataset_type="paired",
    root="../data/",
)
dataset_paired[0]

({'seq': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT',
  'seq_alt': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGACTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT'},
 tensor(-0.2812))

In [15]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="paired",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="paired",transform=rev_transform,root="../data/",)

print("paired info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="paired")

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


paired info
Dataset VaishnavDataset (MpraDaraset)
    Number of datapoints: 2983
    Root location: ../data/Vaishnav
    Using split: test
    Split: {'train': 28214427, 'val': 3134936, 'test native': 3929, 'test drift': 2983, 'test paired': 2983}
    Task: Regression
    Description: The Vaishnav dataset includes measurements of promoter-dependent expression: 30,722,376 for the standard medium (YPD) and 20,616,659 for the minimal medium (SD-Ura). The input data are DNA sequences of variable length, and the outputs are a scalar value of regulatory activity. The data is divided into training and validation sets in a 9:1 ratio. The independent test set contains measurements for native promoters: 3,928 for YPD and 3,977 for SD-Ura.
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 29.71it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 29.39it/s]
paired Pearson correlation


tensor(-0.1636)